# Sentiment Classification Project

In [1]:
import ctypes
ctypes.CDLL("/cluster/data/cuda/12.8.1/lib64/libcudart.so", mode=ctypes.RTLD_GLOBAL)
ctypes.CDLL("/cluster/data/cuda/12.8.1/lib64/libnvrtc.so", mode=ctypes.RTLD_GLOBAL)
ctypes.CDLL("/cluster/data/cuda/12.8.1/lib64/libnvJitLink.so", mode=ctypes.RTLD_GLOBAL)

import sys
sys.path.insert(0, "/home/taekim/.local/lib/python3.14/site-packages")

import torch
print(torch.__file__)
print(torch.version.cuda)
x = torch.randn(3, 3).cuda()
print(x @ x)

/home/taekim/.local/lib/python3.14/site-packages/torch/__init__.py
12.8
tensor([[-0.6413,  0.8083, -0.6229],
        [-1.1512, -0.0799,  1.2800],
        [ 2.5376, -0.1119, -1.0512]], device='cuda:0')


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Verify Setup
Make sure cuda (GPU) is available

In [3]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

CUDA Available: True
Device Name: NVIDIA GeForce RTX 5060 Ti


# Load data

In [4]:
train_full = pd.read_csv("/cluster/courses/cil/text-classification/data/train.csv")

# Data preprocessing TODO
We need to preprocess data. Step that come to my mind:
 - Remove word count outliers. (The vibe of a review comes across after 100 words or so).
 - We have german and english data. Should we translate everything to english
 - Should we search for smiley an insert text for them.
 - Should we search for ** (bold markers) and emphasize this word differenetly? 

# Build Validation Set
Todo: 

Currently we use 90% of the reviews for training, and the remaining 10% for validation
This should be optimized. Use k-fold validation or something like that to get most of our data as training and enable proper hyperparameter tuning later on.

In [5]:
train_df, val_df = train_test_split(
        train_full, test_size=0.1, stratify=train_full["label"], random_state=42
)

# Baselines
The current baseline implementation follows the structure:
$$\text{Output} = \text{Classifier}(\text{EmbeddingForTokens}(x))$$

In this specific case:
* **EmbeddingForTokens**: They use `CountVectorizer`, which creates a "Bag of Words" representation (counting word frequencies).
* **Classifier**: They use **Logistic Regression**.

We could implement additional baselines by choosing more complex **EmbeddingForTokens** methods (such as word-level embeddings or pre-trained vectors) and more sophisticated **Classifier** models (such as Random Forests or simple Neural Networks (MLP)).

# TRAIN PIPELINE

## Overview
A modular training pipeline that evaluates combinations of (vectorizer, classifier) pairs.
Defined across three files: embeddings.py, models.py, and train_loop.py.

## Step 1 — Vectorization (embeddings.py)
For classical vectorizers (BoW, TF-IDF), the signature is:
    (X_train, X_val) -> (X_train_embedded, X_val_embedded)
The vocabulary is fit on X_train only, then applied to both splits.

For pretrained embedding models (BERT, GloVe), sentences are encoded
independently with no fitting step:
    X -> X_embedded

## Step 2 — Classification (models.py)
Each classifier is a factory function that returns a fresh, untrained model instance.
Keeping them as functions (rather than instances) ensures each combination in the
loop starts from a clean state.

model is then used to fit X_train on Y_train: model.fit(X_train, Y_train)

## Step 3 — Train Loop (train_loop.py)
Takes a list of (vectorizer_fn, model_fn) tuples. For each combination it:
  1. Vectorizes the data
  2. Trains the classifier on the training embeddings X_train, Y_train
  3. Evaluates on both train and validation splits

Returns a list of result dicts, one per combination:
    {
        "vectorizer":          str,   # name of the vectorizer function
        "model":               str,   # name of the model factory
        "training_score":      float,
        "training_mae":        float,
        "training_accuracy":   float,
        "validation_score":    float,
        "validation_mae":      float,
        "validation_accuracy": float
    }

In [6]:
# import sys
# import subprocess
# subprocess.run([sys.executable, "-m", "pip", "install", "sentence-transformers"])

In [7]:
import os
os.chdir('/home/taekim/Garnella')

In [8]:
%load_ext autoreload
%autoreload 2


In [9]:
import os
os.environ["PATH"] = "/cluster/courses/cil/envs/envs/text-classification/bin:" + os.environ.get("PATH", "")
import sys
print(sys.executable)

/cluster/courses/cil/envs/envs/text-classification/bin/python


In [14]:
import sys
!{sys.executable} -m pip install --user --no-binary :all: sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 44.2 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for sentencepiece: filename=sentencepiece-0.2.1-cp314-cp314-linux_x86_64.whl size=1387125 sha256=8bb1ae19972c47d2f122293a40077f729c81e335cf2f7111a62e4c8cbc91d7a2
  Stored in directory: /home/taekim/.cache/pip/wheels/e4/2f/f8/759153c174a1c7eeabeff41817283efe46e09ff03b363cbf2e
Successfully built sentencepiece


In [ ]:
# from embeddings import *
# from models import *
# from train_loop_caching import train_loop


# Cell 1: Embedding comparison BASELINE
combinations_embeddings = [
    #(get_tfidf_embeddings, get_logistic_regression),
   # (get_multilingual_embeddings, get_logistic_regression),
    #(get_gemma_embeddings, get_logistic_regression),
    #(get_qwen_embeddings, get_logistic_regression),
]
#results_emb = train_loop(train_df, val_df, combinations_embeddings)
#pd.DataFrame(results_emb).sort_values("validation_score", ascending=False)


/home/taekim/.local/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:


# from embeddings_adv import *

# finetune_gemma(train_df.sample(10000, random_state=0),
#                val_df=val_df.sample(1000, random_state=0),
#                epochs=1, batch_size=128, lr=2e-4, max_seq_length=64,
#                n_train_pairs=10_000, n_eval_pairs=1_000)

# # Clear stale finetuned embeddings from the smoke test
# for suffix in ["_train.npy", "_val.npy"]:
#     p = f"./embedding_cache/gemma_finetuned{suffix}"
#     if os.path.exists(p):
#         os.remove(p)

# # Retrain on full dataset
# finetune_gemma(train_df, val_df=val_df,
#                epochs=3, batch_size=128, lr=2e-4, max_seq_length=64)

In [ ]:
from models_additional import (
     get_xgboost_ev, get_xgboost_mae,
)
from embeddings_adv import *
from train_loop_caching import train_loop


# combinations_gpu = [
#     # (get_gemma_embeddings_seq128, get_xgboost),
#     # (get_gemma_embeddings_seq128, get_xgboost_ev),       # EV decoding variant
#     (get_gemma_embeddings_seq128, get_xgboost_mae),      # regression w/ MAE objective
# ]

combinations_sentiment = [
    (get_tabularisai_sentiment_embeddings, get_xgboost_ev),
    (get_tabularisai_sentiment_embeddings, get_xgboost_mae)
    (get_nlptown_sentiment_embeddings,    get_xgboost_ev),
    (get_nlptown_sentiment_embeddings,    get_xgboost_mae),
    (get_multilingual_e5_embeddings,       get_xgboost_ev),
    (get_multilingual_e5_embeddings,       get_xgboost_mae),
]
results_sentiment = train_loop(train_df, val_df, combinations_sentiment)
pd.DataFrame(results_sentiment).sort_values("validation_score", ascending=False)



Could not extract SentencePiece model from /home/taekim/.cache/huggingface/hub/models--cardiffnlp--twitter-xlm-roberta-base-sentiment/snapshots/f2f1202b1bdeb07342385c3f807f9c07cd8f5cf8/sentencepiece.bpe.model using sentencepiece library due to 
SentencePieceExtractor requires the SentencePiece library but it was not found in your environment. Check out the instructions on the
installation page of its repo: https://github.com/google/sentencepiece#installation and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.
. Falling back to TikToken extractor.


ValueError: `tiktoken` is required to read a `tiktoken` file. Install it with `pip install tiktoken`.